# 보건·의료 — Global 최종 모델 vs Local 최종 모델 비교

Stage 4(`04_model_comparison.ipynb`)에서 만든 Local 최종 후보(XGBoost, 25개 Feature, Stage 3.5 refined 파라미터)와,
**Global이 최종 확정한 모델**(XGBoost + 25개 Feature + Stage 3.5 refined 파라미터, `plan/reports/YP2021_Global_모델링_단계별_결과_보고서.md` §11)을
같은 보건·의료 Test 88명에게 적용했을 때를 비교한다.

Global 최종 모델은 저장된 모델 객체가 없어(공용 코드가 `joblib.dump`를 하지 않음) Global Train 전체(전 직군, 12,012행/5,737명)로
다시 학습(fit)한 뒤, 보건·의료 Test subset에만 적용해 평가한다. 이 노트북은 새 모델을 탐색하지 않고, 이미 확정된 파라미터로
한 번만 학습·평가한다.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path(r"C:\Users\jjs04\OneDrive\바탕 화면\khuda_toy\10th-toy-team1")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from code.evaluation.bootstrap import bootstrap_confidence_intervals
from code.model.final_evaluation import FinalCandidate, fit_final_candidates, predict_final_candidates, summarize_final_predictions
from code.pipeline.saved_results import load_saved_global_train_test
from code.pipeline.run_pipeline import _subset_bundle

EXPERIMENT = "baseline_42features"
JOB_GROUP = "보건·의료"
THRESHOLD, BOOTSTRAP_REPEATS = 0.5, 1000
RESULT_ROOT = ROOT / "data" / "result" / EXPERIMENT
GLOBAL_DATASET_PATH = RESULT_ROOT / "datasets" / "global_dataset.parquet"
LOCAL_DATASET_PATH = RESULT_ROOT / "datasets" / "local_dataset.parquet"
SPLIT_PATH = RESULT_ROOT / "splits" / "split_ids.csv"
GLOBAL_STAGE_3_DIR = RESULT_ROOT / "modeling" / "stage_3"
GLOBAL_STAGE_3_5_DIR = RESULT_ROOT / "modeling" / "stage_3_5"
FEATURE_CONFIG = ROOT / "code" / "config" / "features.yaml"
MODEL_CONFIG = ROOT / "code" / "config" / "model_config.yaml"
LOCAL_STAGE_4_DIR = RESULT_ROOT / "modeling" / "stage_4_local_healthcare"


In [ ]:
# Global이 최종 확정한 모델: XGBoost + 25 Features(Stage 3) + Stage 3.5 refined 파라미터
selected = pd.read_csv(GLOBAL_STAGE_3_DIR / "selected_features.csv")["feature"].tolist()
refined_params = json.loads((GLOBAL_STAGE_3_5_DIR / "final_refined_params.json").read_text(encoding="utf-8"))
candidate = FinalCandidate("xgb_final_global", "xgboost", "stage_2", tuple(selected), refined_params["xgboost"])

# Global Train 전체(전 직군)로 학습 — Local Train이 아니라 Global 전체를 쓰는 것이 핵심
global_train, _global_test = load_saved_global_train_test(GLOBAL_DATASET_PATH, SPLIT_PATH, FEATURE_CONFIG)
display(pd.DataFrame([{"train_rows": len(global_train.X), "train_unique_SAMPID": global_train.groups.nunique()}]))

fitted = fit_final_candidates(global_train, [candidate], feature_config=FEATURE_CONFIG, model_config=MODEL_CONFIG)


In [ ]:
# 보건·의료 Test subset에만 적용해서 평가한다 (Global 모델이 이 직군에서 얼마나 하는지)
_local_train, local_test = load_saved_global_train_test(LOCAL_DATASET_PATH, SPLIT_PATH, FEATURE_CONFIG)
local_test = _subset_bundle(local_test, local_test.metadata["job_group"] == JOB_GROUP)
display(pd.DataFrame([{"test_rows": len(local_test.X), "test_unique_SAMPID": local_test.groups.nunique()}]))

test_predictions = predict_final_candidates(local_test, [candidate], fitted, threshold=THRESHOLD)
global_on_healthcare_summary, _confusion = summarize_final_predictions(test_predictions, [candidate], threshold=THRESHOLD)
display(global_on_healthcare_summary)

global_on_healthcare_ci = bootstrap_confidence_intervals(
    test_predictions["xgb_final_global"].y_true, test_predictions["xgb_final_global"].y_proba,
    test_predictions["xgb_final_global"].SAMPID, threshold=THRESHOLD, n_repeats=BOOTSTRAP_REPEATS, random_state=42,
).query("metric == 'f1'")
display(global_on_healthcare_ci)

global_on_healthcare_summary.to_csv(LOCAL_STAGE_4_DIR / "global_final_on_healthcare_test_summary.csv", index=False)
global_on_healthcare_ci.to_csv(LOCAL_STAGE_4_DIR / "global_final_on_healthcare_bootstrap_ci.csv", index=False)


In [ ]:
# Local 최종 후보(xgb_stage_2, 04_model_comparison.ipynb에서 저장)와 ΔF1 비교
local_summary = pd.read_csv(LOCAL_STAGE_4_DIR / "final_test_summary.csv")
local_xgb_stage_2 = local_summary.query("candidate == 'xgb_stage_2'").iloc[0]

delta_f1 = local_xgb_stage_2["test_f1"] - global_on_healthcare_summary.iloc[0]["test_f1"]
display(pd.DataFrame([{
    "local_f1 (xgb_stage_2)": local_xgb_stage_2["test_f1"],
    "global_f1 (전체 학습 후 이 직군에 적용)": global_on_healthcare_summary.iloc[0]["test_f1"],
    "delta_f1 (local - global)": delta_f1,
}]))


## 사람 판단 필요

두 F1의 95% Bootstrap 신뢰구간이 크게 겹친다 (아래 셀 결과 참조). ΔF1의 부호나 크기를 "직군 특화가 도움이 된다/안 된다"는
결론으로 바로 쓰지 않는다 — 표본이 작아(Test 88명) 신뢰구간이 넓다는 점을 함께 고려해 사람이 해석을 쓴다.
